# ML-09 ? Validation and Research Claim Audit

## 1. Two paper findings + my methodology questions

The March 2026 FlyRank paper reports that growing and declining content differ in depth, age, and visibility, based on portfolio comparisons. Its label is a 30-day trend direction; this supports an observed association, not a refresh-effect claim. A stronger extension would state the bucket sizes and use a future window for a predictive claim.

It also argues to protect page-one assets and interpret composite scores through raw search performance. That is a sensible triage recommendation, but the paper itself describes the composite score as context rather than a market-standard outcome. I would ask whether the recommendation is evaluated on a client/time holdout and whether its top-K benefit exceeds a transparent raw-signal baseline.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy
import os, duckdb, numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, average_precision_score
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets or the HF_TOKEN environment variable first."
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
pages = con.sql(f"""
SELECT client_hash_id, content_hash_id,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_prev,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_prev,
 AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS avg_position_prev,
 STDDEV(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS position_volatility_prev,
 COUNT(CASE WHEN report_date < DATE '2026-03-16' AND gsc_impressions > 0 THEN 1 END) AS active_days_prev,
 SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_outcome
FROM {FACT} GROUP BY 1,2 HAVING impressions_prev >= 100
""").df()
pages["is_declining"] = (pages.impressions_outcome < 0.8 * pages.impressions_prev).astype(int)
features=["impressions_prev","clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
pages["log_impressions_prev"]=np.log1p(pages.impressions_prev); pages["log_clicks_prev"]=np.log1p(pages.clicks_prev)
features=["log_impressions_prev","log_clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
def p_at_k(y,s,k):
 k=min(k,len(y)); return float(pd.DataFrame({"y":np.asarray(y),"s":np.asarray(s)}).nlargest(k,"s").y.mean())
def baseline_score(d):
 return d.impressions_prev * (1 + (d.position_volatility_prev.fillna(0) >= 5).astype(int))
print(f"Page-level March frame: {len(pages):,} rows; positive rate: {pages.is_declining.mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level March frame: 77,540 rows; positive rate: 28.5%


## 2. My model under an honest split (before/after)

I report a deliberately easier random row split beside the client-held-out split. The comparison exposes whether random splitting inflated the apparent ranking quality; only the grouped result supports the cross-client claim.

In [2]:
from sklearn.model_selection import train_test_split
# Random row split: diagnostic only.
rtr,rte=train_test_split(np.arange(len(pages)),test_size=.25,random_state=42,stratify=pages.is_declining)
random_model=Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,min_samples_leaf=25,class_weight="balanced_subsample",random_state=42,n_jobs=-1))])
random_model.fit(pages.iloc[rtr][features],pages.iloc[rtr].is_declining)
random_p=p_at_k(pages.iloc[rte].is_declining,random_model.predict_proba(pages.iloc[rte][features])[:,1],50)
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
tr,te=next(splitter.split(pages[features],pages.is_declining,groups=pages.client_hash_id))
train,test=pages.iloc[tr].copy(),pages.iloc[te].copy()
assert not set(train.client_hash_id).intersection(test.client_hash_id)
models={
 "logistic_regression":Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))]),
 "random_forest":Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,min_samples_leaf=25,class_weight="balanced_subsample",random_state=42,n_jobs=-1))])}
rows=[]; fitted={}
for name,model in models.items():
 model.fit(train[features],train.is_declining); score=model.predict_proba(test[features])[:,1]; fitted[name]=model
 rows.append({"method":name,"precision@20":p_at_k(test.is_declining,score,20),"precision@50":p_at_k(test.is_declining,score,50),"average_precision":average_precision_score(test.is_declining,score)})
bscore=baseline_score(test)
rows.insert(0,{"method":"week_4_baseline","precision@20":p_at_k(test.is_declining,bscore,20),"precision@50":p_at_k(test.is_declining,bscore,50),"average_precision":average_precision_score(test.is_declining,bscore)})
comparison=pd.DataFrame(rows).sort_values("precision@50",ascending=False).reset_index(drop=True)
comparison
grouped_p=float(comparison.loc[comparison.method=="random_forest","precision@50"].iloc[0])
pd.DataFrame({"validation":["random_row_holdout (diagnostic)","client_group_holdout (reported)"],"precision@50":[random_p,grouped_p],"base_rate":[float(pages.iloc[rte].is_declining.mean()),float(test.is_declining.mean())]})


,validation,precision@50,base_rate
0,random_row_holdout (diagnostic),0.80,0.285478
1,client_group_holdout (reported),0.18,0.147470


## 3. Leakage audit

Timeline: March 1?15 features ? March 16?31 label. The final feature set excludes `impressions_outcome`, the label `is_declining`, both identifiers, and any product score/flag. The audit below intentionally adds the outcome-window total only as a test; its score must not be used as a model result.

In [3]:
leaky_features=features+["impressions_outcome"]
leaky=Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,min_samples_leaf=25,random_state=42,n_jobs=-1))])
leaky.fit(train[leaky_features],train.is_declining)
leaky_p=p_at_k(test.is_declining,leaky.predict_proba(test[leaky_features])[:,1],50)
forbidden={"impressions_outcome","is_declining","client_hash_id","content_hash_id"}
assert not forbidden.intersection(features)
print("Honest feature set:",features)
print("Leakage demonstration Precision@50 (DO NOT REPORT AS MODEL PERFORMANCE):",round(leaky_p,3))
print("PASS: final feature set has no identifiers, label, or outcome-window input.")


Honest feature set: ['log_impressions_prev', 'log_clicks_prev', 'avg_position_prev', 'position_volatility_prev', 'active_days_prev']
Leakage demonstration Precision@50 (DO NOT REPORT AS MODEL PERFORMANCE): 1.0
PASS: final feature set has no identifiers, label, or outcome-window input.


## 4. Claim rewrite

**Too bold:** ?The model identifies pages that should be refreshed and will recover.?

**Evidence-carrying version:** ?On this March development slice, the grouped client holdout measures how well the model ranks pages that later meet the defined short-window decline proxy. The score is decision-support for human review; it does not show that refreshing a page will cause recovery.?

In [4]:
audit={"random_row_precision_at_50":float(random_p),"grouped_precision_at_50":float(grouped_p),"leakage_demo_precision_at_50":float(leaky_p),"final_features":features,"seed":42}
Path("work/outputs").mkdir(parents=True,exist_ok=True)
Path("work/outputs/w06_validation_audit.json").write_text(json.dumps(audit,indent=2))
print("Saved safe validation receipt: work/outputs/w06_validation_audit.json")


Saved safe validation receipt: work/outputs/w06_validation_audit.json


## Self-check

- [x] Random and client-grouped results are separated
- [x] Base rates appear beside Precision@50
- [x] Deliberate leakage test is labelled non-reportable
- [x] Claim distinguishes observed ranking from causal refresh impact